In [ ]:
!uv pip install kagglehub -qU
import kagglehub
#path1 = kagglehub.dataset_download("manith04/ddd-processed-1-training-frames-type-1")
path2 = kagglehub.dataset_download("manith04/ddd-processed-2-training-frames-type-1")
# path3 = kagglehub.dataset_download("manith04/ddd-processed-validation-frames-type-1",)



100%|██████████| 3.37G/3.37G [00:37<00:00, 95.5MB/s]

Extracting files...


In [ ]:
print(path2)

/root/.cache/kagglehub/datasets/manith04/ddd-processed-2-training-frames-type-1/versions/1


In [ ]:
!uv pip install mediapipe -q

In [ ]:
%%file run.py

import os
os.environ.pop("MPLBACKEND", None)
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ---------------- CONFIG ----------------

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

TARGET_SIZE = 64
MARGIN = 0.3  # fraction of eye size to expand the crop

MODEL_PATH = "/content/face_landmarker.task"  # <-- change this

# ---------------- EYE CROP FUNCTION ----------------

def crop_eye_exact64(image, landmarks, eye_indices):
    h, w, _ = image.shape

    # Get eye landmark points
    pts = np.array([
        [int(landmarks[i].x * w), int(landmarks[i].y * h)]
        for i in eye_indices
    ])

    # Compute bounding box
    x, y, bw, bh = cv2.boundingRect(pts)
    cx, cy = x + bw // 2, y + bh // 2

    # Expand bounding box by margin
    size = int(max(bw, bh) * (1 + MARGIN))

    # Crop coordinates
    x1 = cx - size // 2
    y1 = cy - size // 2
    x2 = x1 + size
    y2 = y1 + size

    # Pad if coordinates go out of image bounds
    pad_left = max(-x1, 0)
    pad_top = max(-y1, 0)
    pad_right = max(x2 - w, 0)
    pad_bottom = max(y2 - h, 0)

    # Apply padding if necessary
    x1 = max(x1, 0)
    y1 = max(y1, 0)
    x2 = min(x2, w)
    y2 = min(y2, h)

    eye = image[y1:y2, x1:x2]

    # Pad to make it exactly size x size
    if pad_top > 0 or pad_bottom > 0 or pad_left > 0 or pad_right > 0:
        eye = cv2.copyMakeBorder(
            eye,
            pad_top, pad_bottom, pad_left, pad_right,
            borderType=cv2.BORDER_REPLICATE
        )

    # Final check
    if eye.shape[0] != TARGET_SIZE or eye.shape[1] != TARGET_SIZE:
        eye = cv2.resize(eye, (TARGET_SIZE, TARGET_SIZE))  # fallback

    return eye

# ---------------- PROCESS SINGLE IMAGE ----------------

def process_image(image_path, input_root, output_root, detector):
    image = cv2.imread(str(image_path))
    if image is None:
        return

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    result = detector.detect(mp_image)
    if not result.face_landmarks:
        return

    landmarks = result.face_landmarks[0]

    left_eye = crop_eye_exact64(image, landmarks, LEFT_EYE)
    right_eye = crop_eye_exact64(image, landmarks, RIGHT_EYE)

    # Preserve original relative path
    rel_path = image_path.relative_to(input_root)

    left_save_path = output_root / "left_eye" / rel_path
    right_save_path = output_root / "right_eye" / rel_path

    # Create parent directories
    left_save_path.parent.mkdir(parents=True, exist_ok=True)
    right_save_path.parent.mkdir(parents=True, exist_ok=True)

    cv2.imwrite(str(left_save_path), left_eye)
    cv2.imwrite(str(right_save_path), right_eye)

# ---------------- PROCESS DATASET ----------------

def process_dataset(input_root, output_root):
    input_root = Path(input_root)
    output_root = Path(output_root)

    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1
    )

    with vision.FaceLandmarker.create_from_options(options) as detector:
        image_paths = list(input_root.rglob("*.jpg"))
        for img_path in tqdm(image_paths):
            process_image(img_path, input_root, output_root, detector)

# ---------------- RUN ----------------

INPUT_IMAGES = "/root/.cache/kagglehub/datasets/manith04/ddd-processed-2-training-frames-type-1/versions/1"
OUTPUT_EYES = "/content/processed_training_1_eyes_exact64"

process_dataset(INPUT_IMAGES, OUTPUT_EYES)

Writing run.py


In [ ]:
!wget -O /content/face_landmarker.task \
https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

--2026-03-06 07:19:08--  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.121.207, 192.178.142.207, 142.251.108.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.121.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3758596 (3.6M) [application/octet-stream]
Saving to: ‘/content/face_landmarker.task’

/content/face_landm 100%[===================>]   3.58M  --.-KB/s    in 0.01s   

2026-03-06 07:19:08 (255 MB/s) - ‘/content/face_landmarker.task’ saved [3758596/3758596]



In [ ]:
!python run.py

2026-03-06 07:19:12.533925: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-06 07:19:12.901263: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-06 07:19:13.098660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772781553.359705    1482 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772781553.433149    1482 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772781553.979333    1482 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
import os
os.environ["KAGGLE_USERNAME"] = "manith04"
os.environ["KAGGLE_API_TOKEN"] = "KGAT_28e717b9ccd1d5f6908afdf6559d8cb4"

In [ ]:
!mkdir -p ~/.kaggle
!echo '{"username":"manith04","key":"KGAT_28e717b9ccd1d5f6908afdf6559d8cb4"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
%cd /content/processed_training_1_eyes_exact64/left_eye

/content/processed_training_1_eyes_exact64/left_eye


In [ ]:
!kaggle datasets init -p .

Data package template written to: ./dataset-metadata.json


In [ ]:
!kaggle datasets create -p . --dir-mode zip --public

Starting upload for file 024.zip
100% 48.3M/48.3M [00:01<00:00, 47.1MB/s]
Upload successful: 024.zip (48MB)
Starting upload for file 023.zip
100% 47.3M/47.3M [00:00<00:00, 49.7MB/s]
Upload successful: 023.zip (47MB)
Starting upload for file 033.zip
100% 38.2M/38.2M [00:00<00:00, 40.6MB/s]
Upload successful: 033.zip (38MB)
Starting upload for file 035.zip
100% 47.6M/47.6M [00:01<00:00, 47.5MB/s]
Upload successful: 035.zip (48MB)
Starting upload for file 031.zip
100% 51.0M/51.0M [00:01<00:00, 46.3MB/s]
Upload successful: 031.zip (51MB)
Starting upload for file 032.zip
100% 40.2M/40.2M [00:01<00:00, 41.3MB/s]
Upload successful: 032.zip (40MB)
Starting upload for file 034.zip
100% 55.6M/55.6M [00:01<00:00, 52.1MB/s]
Upload successful: 034.zip (56MB)
Starting upload for file 036.zip
100% 55.2M/55.2M [00:01<00:00, 48.2MB/s]
Upload successful: 036.zip (55MB)
Your public Dataset is being created. Please check progress at https://www.kaggle.com/datasets/manith04/ddd-training2-lefteye


In [ ]:
!uv pip install kagglehub -qU
import kagglehub
path1 = kagglehub.dataset_download("manith04/ddd-processed-1-training-frames-type-1")
#path2 = kagglehub.dataset_download("manith04/ddd-processed-2-training-frames-type-1")
# path3 = kagglehub.dataset_download("manith04/ddd-processed-validation-frames-type-1",)



Using Colab cache for faster access to the 'ddd-processed-1-training-frames-type-1' dataset.


In [ ]:
!ls  /root/.cache/

matplotlib  node-gyp  uv
